# DocuMind - cheques: detector (YOLO) from the labelled boxes to the checked values

Same idea as `train_receipts_yolo.ipynb`: the notebook only calls the package (`documind/`), run it top to bottom with the kernel of your `ai_env`.

```
data/processed/cheques_yolo (prepared)  ->  [documind.training.train_cheques]  ->  models/cheques_yolo.pt
picture  ->  [documind.inference.cheques: detector + Tesseract on each box + checks]  ->  bank, cheque number, payee, amount, date, account, MICR, signature
```
Make the data once, in a terminal (the notebook never rebuilds it):
```
python scripts/generate_cheques.py --n 10000 --out data/raw/cheques
python -m documind.data.prepare_cheques          # then LOOK at reports/cheques_preview/
```
Classes (read from `data/processed/cheques_yolo/data.yaml`): `bank_name, cheque_number, amount_words, payee, amount_digits, place, date, signature, account_number, micr_line`.
Training needs a GPU to be comfortable; on a CPU it works but takes hours. The receipt and cheque models are independent files: training one never touches the other.

In [ ]:
import json, os, sys
from pathlib import Path
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display
from PIL import Image
# Make the project root importable when the notebook is opened from a subfolder.
ROOT = Path.cwd()
while not (ROOT / "documind").is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if (ROOT / "documind").is_dir():
    os.chdir(ROOT)
    if str(ROOT) not in sys.path:
        sys.path.insert(0, str(ROOT))
    print(f"project root: {ROOT}")
else:
    raise FileNotFoundError(
        "Could not find the project root containing 'documind/'. "
        "Open the notebook from the repo root or install the project with: %pip install -e ."
    )

from documind.config import MODELS_DIR, PROCESSED_DIR, RAW_DIR, REPORTS_DIR

If a package is missing (first run): `pip install -r requirements-receipts.txt` in a terminal with the `ai_env` activated (Ultralytics brings PyTorch: big download). Cheques use the same requirements as receipts.

## GPU check (run this first)
Ultralytics only uses the GPU if PyTorch is a **CUDA build** and the NVIDIA driver works. `pip install ultralytics` often installs the CPU-only PyTorch on Windows, so check before training.

If the check below stops with an error, fix it in a terminal with `ai_env` activated, then **restart the kernel**:
```
pip uninstall -y torch torchvision torchaudio
pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
```

In [ ]:
import torch

REQUIRE_GPU = True   # True = stop here if no GPU is found (instead of silently training for hours on the CPU)

print("python executable :", sys.executable)          # must be inside your ai_env
print("torch             :", torch.__version__)       # ending with +cpu = CPU-only build
print("torch CUDA build  :", torch.version.cuda)      # None = CPU-only build
print("cuda available    :", torch.cuda.is_available())

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"gpu               : {props.name} ({props.total_memory / 1024**3:.1f} GB VRAM)")
    DEVICE = 0
else:
    DEVICE = "cpu"
    msg = ("No usable GPU for PyTorch. Check, in this order:\n"
           "  1. torch version ends with '+cpu' / CUDA build is None -> reinstall the CUDA build (cell above)\n"
           "  2. python executable is not your ai_env -> Kernel > Change kernel\n"
           "  3. 'nvidia-smi' fails in a terminal -> update the NVIDIA driver")
    if REQUIRE_GPU:
        raise RuntimeError(msg)
    print("WARNING (training will run on the CPU):\n" + msg)

## Settings

In [ ]:
from documind.data.prepare_cheques import CHEQUE_NAMES
from documind.data.prepare_receipts import read_names_from_yaml
from documind.training import train_cheques, train_receipts   # cheques reuse the receipt trainer with their own Profile (paths, imgsz, augmentation)

PROFILE = train_cheques.profile()
CLASS_NAMES = read_names_from_yaml(PROFILE.data_yaml) or CHEQUE_NAMES   # class ids 0..N-1, read from the prepared data
print(len(CLASS_NAMES), "classes:", CLASS_NAMES)

MODEL = PROFILE.model     # yolo11n (fastest) < yolo11s < yolo11m (most accurate, more GPU memory)
EPOCHS = PROFILE.epochs   # 60; the cheque classes are big boxes at fixed places and are learned quickly. 100 = all epochs (also set PATIENCE = 100)
TARGET = 0.90             # score to reach on val AND test (mAP50), checked after the evaluation
IMGSZ = PROFILE.imgsz     # 1280: a cheque is wide (1600 px) and the handwriting / MICR line are small
BATCH = PROFILE.batch     # 4 fits an 8 GB GPU at 1280 with yolo11s; lower it (2) on 'CUDA out of memory'; -1 = automatic
PATIENCE = PROFILE.patience   # stop after this many epochs without progress (best.pt is still the best epoch)
NAME = "cheques"
# DEVICE (0 = first GPU, or "cpu") was set by the GPU check above
WORKERS = max(2, min(8, os.cpu_count() or 2))   # data-loader processes
CACHE = False             # "ram" or "disk" decode every picture once (needs several GB): faster epochs
FRACTION = 1.0            # 0.3 = train on 30 % of the pictures: quick run to compare settings

## Look at the data
The data was prepared in an earlier run (`python -m documind.data.prepare_cheques`). If the boxes in the pictures do not sit on the right things, **stop here**.

In [ ]:
DATA_ROOT = PROFILE.data_yaml.parent
DATA_YAML = PROFILE.data_yaml
assert DATA_YAML.exists(), "prepared data not found: run  python -m documind.data.prepare_cheques  once in a terminal"

counts = {}
for split in ("train", "val", "test"):
    n = [0] * len(CLASS_NAMES)
    for p in (DATA_ROOT / split / "labels").glob("*.txt"):
        for line in p.read_text().splitlines():
            parts = line.split()
            if len(parts) < 5:
                continue
            try:
                cls_id = int(float(parts[0]))
            except ValueError:
                continue
            if 0 <= cls_id < len(n):
                n[cls_id] += 1
    counts[split] = n
df = pd.DataFrame(counts, index=CLASS_NAMES)
display(df)
df.plot(kind="bar", figsize=(11, 4), title="boxes per class and split"); plt.tight_layout(); plt.show()
prev = sorted((REPORTS_DIR / "cheques_preview").glob("*.png"))[:4]
if prev:
    fig, ax = plt.subplots(len(prev), 1, figsize=(9, 4.2 * len(prev)))
    for a, p in zip(ax if len(prev) > 1 else [ax], prev):
        a.imshow(Image.open(p)); a.set_title(p.stem[:40], fontsize=8); a.axis("off")
    plt.show()

## STEP 2 - train
Augmentation chosen for cheques (see `documind/training/train_cheques.py`): no flips (a mirrored cheque is not a cheque), small rotation / perspective (scans and phone pictures), lighter mosaic (0.5) so that the fields keep their relative places, brightness changes.

The same trainer as the receipts (`train_receipts.train`), given the cheque `Profile`: its paths (`data/processed/cheques_yolo`, `models/cheques_yolo.pt`, `reports/cheques_runs`), its augmentation and its messages. The first lines of the training log must show `CUDA:0 (NVIDIA GeForce RTX 3080 ...)`.

**Speed check while an epoch runs:** open Task Manager > Performance > GPU (look at the *CUDA* or *Compute* graph) or run `nvidia-smi -l 1` in a terminal.
Below about 70 % the GPU is waiting for the data loader: raise `WORKERS` or set `CACHE = "ram"` / `"disk"`.
Above 90 % it is the GPU: use `MODEL = "yolo11n.pt"` or `IMGSZ = 800` and compare with `FRACTION = 0.3` runs first.

In [ ]:
import inspect
import os

import torch

def _accepts(fn, arg):
    params = inspect.signature(fn).parameters
    return arg in params or any(p.kind is inspect.Parameter.VAR_KEYWORD for p in params.values())

if "DEVICE" not in globals():
    DEVICE = 0 if torch.cuda.is_available() else "cpu"

train_kwargs = dict(
    data=PROFILE.data_yaml,
    model=MODEL,
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    patience=PATIENCE,
    name=NAME,
    workers=WORKERS,
    cache=CACHE,
    fraction=FRACTION,
    profile=PROFILE,
)
if _accepts(train_receipts.train, "device"):
    train_kwargs["device"] = DEVICE

weights = train_receipts.train(**train_kwargs)
print("best weights:", weights)

In [ ]:
run_dir = PROFILE.runs_dir / NAME
for f in ("results.png", "confusion_matrix_normalized.png"):
    path = run_dir / f
    if path.exists():
        plt.figure(figsize=(11, 6))
        plt.imshow(Image.open(path))
        plt.axis("off")
        plt.title(f)
        plt.show()

## Evaluate on val AND test
`test` was never used for training or for choosing the best epoch. Read the **per-class** rows: a good average can hide a class that never works (the `signature` box and the long handwritten `amount_words` / `payee` lines are the usual suspects).

`found_rate` = is the most confident box of that class on the labelled box (IoU >= 0.5)? Every cheque has exactly one box of each class, so it is the number closest to what the user sees; `mAP50` is the strict one.

**Be careful with what a high score means:** the cheques come from `generate_cheques.py`, and val / test come from the same generator as train. A score close to 1.0 says the model learned the generated layouts, not that it reads every real cheque. Try real cheques (STEP 3) before trusting it.

In [ ]:
eval_kwargs = dict(imgsz=IMGSZ, profile=PROFILE)
if _accepts(train_receipts.evaluate, "device"):
    eval_kwargs["device"] = DEVICE
rows = train_receipts.evaluate(weights, PROFILE.data_yaml, **eval_kwargs)
display(pd.DataFrame(rows).round(3))
# did we reach the target (val AND test)?  mAP50 is the strict number, found_rate the one closer to what the user sees
for metric in ("mAP50", "found_rate"):
    print(f"--- {metric}")
    train_receipts.print_verdict(train_receipts.check_target(rows, TARGET, metric), PROFILE.hints, PROFILE.reached_note)

## If the score is below the target
The labels of `generate_cheques.py` are exact by construction, so there is no label audit step (that is for hand-labelled data). In order of effect: look at the weakest class above and at the confusion matrix; a bigger model (`MODEL = "yolo11m.pt"`, `BATCH = 2`); all the epochs (`EPOCHS = 100`, `PATIENCE = 100`); more pictures (`python scripts/generate_cheques.py --n 20000 --seed 1`, then prepare again).

## See the predictions on test pictures

In [ ]:
from ultralytics import YOLO
model = YOLO(str(weights))
test_imgs = sorted((DATA_ROOT / "test" / "images").glob("*.jpg"))[:4]
res = model.predict(source=[str(p) for p in test_imgs], imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)
fig, ax = plt.subplots(len(res), 1, figsize=(9, 4.2 * len(res)))
for a, r in zip(ax if len(res) > 1 else [ax], res):
    a.imshow(r.plot()[..., ::-1]); a.axis("off")
plt.show()

## STEP 3 - from boxes to checked values
The detector says **where** each field is; Tesseract reads **what** is written there (`documind/inference/cheques.py`), then the values are parsed and **checked**: amount in figures = amount in words (the French words are converted back to a number), cheque number = the one in the MICR line, valid date, account number length, signature found. A failed check is a warning, never a silent correction.

Tesseract reads printed text and clean handwriting-like fonts well and real cursive handwriting badly: for real cheques replace `read_text` in `documind/inference/cheques.py` by a handwriting model (e.g. TrOCR).

In [ ]:
from documind.inference.cheques import extract_cheque_bytes
for p in test_imgs[:3]:
    r = extract_cheque_bytes(p.read_bytes(), p.name, model=model)
    print(p.name)
    print(json.dumps({"row": r["row"], "checks": r["checks"], "warnings": [w["message"] for w in r["warnings"]]}, ensure_ascii=False, indent=2)[:1800], "\n")

### How often does each check pass? (30 test pictures)
The generated cheques are all consistent, so a `mismatch` here is an OCR mistake, not a bad cheque. `not checked` = the words could not be read as a number.

In [ ]:
from collections import Counter
all_imgs = sorted((DATA_ROOT / "test" / "images").glob("*.jpg"))[:30]
results = [extract_cheque_bytes(p.read_bytes(), p.name, model=model) for p in all_imgs]
print("amount in figures vs in words:", dict(Counter(r["row"]["AMOUNT_CHECK"] for r in results)))
print("cheque number vs MICR line   :", dict(Counter(str(r["checks"].get("micr_number_match")) for r in results)))
print("valid date                   :", dict(Counter(str(r["checks"].get("date_valid")) for r in results)))
print("signature found              :", dict(Counter(str(r["checks"].get("signature_present")) for r in results)))
print("warnings                     :", dict(Counter(w["code"] for r in results for w in r["warnings"])))

### Notes
- The model is saved in `models/cheques_yolo.pt`, the scores in `reports/cheques_report.json` (both are git-ignored like the rest).
- **GPU memory:** the RTX 3080 Laptop has 8 or 16 GB. With `yolo11s`, `IMGSZ = 1280`, `BATCH = 4` it should fit; on `CUDA out of memory` lower `BATCH` to 2.
- **Licence:** Ultralytics YOLO is AGPL-3.0 (a commercial licence exists): fine for a university project, read it before a commercial use.
- Command-line version of the same steps: `python -m documind.training.train_cheques`, `python -m documind.inference.cheques cheque.jpg`.